<a href="https://colab.research.google.com/github/sharmila0510-xx/COMPLAINT_MANAGEMENT_SYSTEM/blob/main/compliant_management_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip -q install flask pyngrok

In [6]:
import os
import sqlite3
import threading
import time
from flask import Flask, request, jsonify, render_template_string


PROJECT_DIR = "/content/campus_complaint_system"
os.makedirs(PROJECT_DIR, exist_ok=True)

DB_PATH = os.path.join(PROJECT_DIR, "campus_complaints.db")

app = Flask(__name__)


def get_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_database():

    conn = get_db()
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS complaints (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            complaint_id TEXT UNIQUE,
            student_name TEXT NOT NULL,
            student_email TEXT NOT NULL,
            department TEXT NOT NULL,
            year TEXT NOT NULL,
            category TEXT NOT NULL,
            priority TEXT NOT NULL,
            location TEXT NOT NULL,
            description TEXT NOT NULL,
            status TEXT NOT NULL DEFAULT 'Submitted',
            admin_remark TEXT DEFAULT '',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    conn.commit()
    conn.close()


init_database()


HTML = r"""
<!DOCTYPE html>
<html lang="en">

<head>

<meta charset="UTF-8">

<meta name="viewport" content="width=device-width, initial-scale=1.0">

<title>CampusCare | Campus Complaint Management</title>

<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>

<style>

* {
    box-sizing: border-box;
    margin: 0;
    padding: 0;
}

body {
    font-family: Arial, Helvetica, sans-serif;
    background: #f4f7fb;
    color: #1f2937;
}

/* NAVBAR */

.navbar {
    background: #172554;
    color: white;
    padding: 16px 30px;
    display: flex;
    justify-content: space-between;
    align-items: center;
    position: sticky;
    top: 0;
    z-index: 100;
}

.logo {
    font-size: 22px;
    font-weight: bold;
}

.logo span {
    color: #60a5fa;
}

.nav-buttons {
    display: flex;
    gap: 10px;
}

.nav-btn {
    border: none;
    background: transparent;
    color: white;
    padding: 9px 16px;
    border-radius: 7px;
    cursor: pointer;
    font-size: 14px;
}

.nav-btn:hover,
.nav-btn.active {
    background: #2563eb;
}

/* MAIN */

.container {
    max-width: 1200px;
    margin: auto;
    padding: 30px 20px;
}

.page {
    display: none;
}

.page.active {
    display: block;
}

/* HERO */

.hero {
    background: linear-gradient(135deg, #1d4ed8, #172554);
    color: white;
    padding: 45px;
    border-radius: 18px;
    margin-bottom: 25px;
}

.hero h1 {
    font-size: 34px;
    margin-bottom: 12px;
}

.hero p {
    color: #dbeafe;
    max-width: 700px;
    line-height: 1.6;
}

/* CARDS */

.card {
    background: white;
    border-radius: 14px;
    padding: 25px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.06);
    margin-bottom: 20px;
}

.card h2 {
    margin-bottom: 20px;
}

/* FORM */

.form-grid {
    display: grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 18px;
}

.form-group {
    display: flex;
    flex-direction: column;
}

.form-group.full {
    grid-column: 1 / -1;
}

label {
    font-size: 14px;
    font-weight: bold;
    margin-bottom: 7px;
}

input,
select,
textarea {
    padding: 12px;
    border: 1px solid #d1d5db;
    border-radius: 8px;
    font-size: 14px;
    outline: none;
}

input:focus,
select:focus,
textarea:focus {
    border-color: #2563eb;
}

textarea {
    min-height: 120px;
    resize: vertical;
}

.primary-btn {
    background: #2563eb;
    color: white;
    border: none;
    padding: 12px 22px;
    border-radius: 8px;
    cursor: pointer;
    font-weight: bold;
}

.primary-btn:hover {
    background: #1d4ed8;
}

/* STATISTICS */

.stats {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 18px;
    margin-bottom: 25px;
}

.stat {
    background: white;
    padding: 22px;
    border-radius: 14px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.06);
}

.stat-title {
    color: #6b7280;
    font-size: 13px;
    margin-bottom: 8px;
}

.stat-value {
    font-size: 30px;
    font-weight: bold;
    color: #172554;
}

/* TABLE */

.table-container {
    overflow-x: auto;
}

table {
    width: 100%;
    border-collapse: collapse;
}

th,
td {
    padding: 13px;
    border-bottom: 1px solid #e5e7eb;
    text-align: left;
    font-size: 13px;
}

th {
    background: #f8fafc;
}

tr:hover {
    background: #f8fafc;
}

/* STATUS */

.status {
    display: inline-block;
    padding: 5px 9px;
    border-radius: 20px;
    font-size: 11px;
    font-weight: bold;
}

.status-submitted {
    background: #dbeafe;
    color: #1d4ed8;
}

.status-progress {
    background: #fef3c7;
    color: #92400e;
}

.status-resolved {
    background: #dcfce7;
    color: #166534;
}

.status-rejected {
    background: #fee2e2;
    color: #991b1b;
}

/* FILTER */

.filters {
    display: grid;
    grid-template-columns: 2fr 1fr 1fr;
    gap: 12px;
    margin-bottom: 20px;
}

/* TRACK */

.track-result {
    margin-top: 20px;
}

.timeline {
    border-left: 3px solid #2563eb;
    padding-left: 20px;
}

.timeline-item {
    margin-bottom: 18px;
}

.timeline-item h4 {
    margin-bottom: 5px;
}

.timeline-item p {
    color: #6b7280;
    font-size: 13px;
}

/* LOGIN */

.login-box {
    max-width: 450px;
    margin: 40px auto;
}

.error {
    background: #fee2e2;
    color: #991b1b;
    padding: 12px;
    border-radius: 8px;
    margin-bottom: 15px;
    display: none;
}

.success {
    background: #dcfce7;
    color: #166534;
    padding: 15px;
    border-radius: 8px;
    margin-bottom: 20px;
    display: none;
}

/* CHART */

.chart-container {
    max-width: 600px;
    margin: auto;
}

/* FOOTER */

footer {
    background: #172554;
    color: #cbd5e1;
    text-align: center;
    padding: 25px;
    margin-top: 40px;
    font-size: 13px;
}

/* RESPONSIVE */

@media(max-width: 800px) {

    .form-grid {
        grid-template-columns: 1fr;
    }

    .form-group.full {
        grid-column: auto;
    }

    .stats {
        grid-template-columns: repeat(2, 1fr);
    }

    .filters {
        grid-template-columns: 1fr;
    }

    .navbar {
        flex-direction: column;
        gap: 12px;
    }

    .hero {
        padding: 30px 22px;
    }

    .hero h1 {
        font-size: 27px;
    }

}

</style>

</head>

<body>

<!-- NAVBAR -->

<nav class="navbar">

<div class="logo">
Campus<span>Care</span>
</div>

<div class="nav-buttons">

<button class="nav-btn active" onclick="showPage('home', this)">
Home
</button>

<button class="nav-btn" onclick="showPage('submit', this)">
Submit Complaint
</button>

<button class="nav-btn" onclick="showPage('track', this)">
Track Complaint
</button>

<button class="nav-btn" onclick="showPage('adminLogin', this)">
Admin
</button>

</div>

</nav>


<div class="container">


<!-- HOME -->

<section id="home" class="page active">

<div class="hero">

<h1>Campus Complaint Management System</h1>

<p>
A cloud-based platform for students to submit, track and manage
campus complaints efficiently. The system helps administrators
monitor complaints and improve campus services.
</p>

</div>


<div class="stats">

<div class="stat">
<div class="stat-title">Total Complaints</div>
<div class="stat-value" id="homeTotal">0</div>
</div>

<div class="stat">
<div class="stat-title">Submitted</div>
<div class="stat-value" id="homeSubmitted">0</div>
</div>

<div class="stat">
<div class="stat-title">In Progress</div>
<div class="stat-value" id="homeProgress">0</div>
</div>

<div class="stat">
<div class="stat-title">Resolved</div>
<div class="stat-value" id="homeResolved">0</div>
</div>

</div>


<div class="card">

<h2>How It Works</h2>

<div class="form-grid">

<div>
<h3>1. Submit</h3>
<p>Students submit their campus complaint through the online form.</p>
</div>

<div>
<h3>2. Review</h3>
<p>Administrators review and assign the complaint for action.</p>
</div>

<div>
<h3>3. Track</h3>
<p>Students can track their complaint using the complaint ID.</p>
</div>

<div>
<h3>4. Resolve</h3>
<p>Administrators update the complaint when the issue is resolved.</p>
</div>

</div>

</div>

</section>


<!-- SUBMIT -->

<section id="submit" class="page">

<div class="card">

<h2>Submit a Campus Complaint</h2>

<div id="submitSuccess" class="success"></div>

<div id="submitError" class="error"></div>

<form id="complaintForm">

<div class="form-grid">

<div class="form-group">

<label>Student Name</label>

<input
type="text"
id="student_name"
required
placeholder="Enter your name">

</div>


<div class="form-group">

<label>Student Email</label>

<input
type="email"
id="student_email"
required
placeholder="example@college.edu">

</div>


<div class="form-group">

<label>Department</label>

<select id="department" required>

<option value="">Select Department</option>

<option>CSE</option>
<option>ECE</option>
<option>EEE</option>
<option>MECH</option>
<option>CIVIL</option>
<option>IT</option>
<option>AI & DS</option>
<option>MBA</option>
<option>Other</option>

</select>

</div>


<div class="form-group">

<label>Year</label>

<select id="year" required>

<option value="">Select Year</option>

<option>1st Year</option>
<option>2nd Year</option>
<option>3rd Year</option>
<option>4th Year</option>

</select>

</div>


<div class="form-group">

<label>Complaint Category</label>

<select id="category" required>

<option value="">Select Category</option>

<option>Classroom</option>
<option>Hostel</option>
<option>Electrical</option>
<option>Internet / Wi-Fi</option>
<option>Cleanliness</option>
<option>Water / Plumbing</option>
<option>Transport</option>
<option>Library</option>
<option>Laboratory</option>
<option>Other</option>

</select>

</div>


<div class="form-group">

<label>Priority</label>

<select id="priority" required>

<option value="">Select Priority</option>

<option>Low</option>
<option>Medium</option>
<option>High</option>
<option>Critical</option>

</select>

</div>


<div class="form-group full">

<label>Location</label>

<input
type="text"
id="location"
required
placeholder="Example: Block A, Room 204">

</div>


<div class="form-group full">

<label>Complaint Description</label>

<textarea
id="description"
required
placeholder="Describe the problem clearly"></textarea>

</div>

</div>

<br>

<button class="primary-btn" type="submit">
Submit Complaint
</button>

</form>

</div>

</section>


<!-- TRACK -->

<section id="track" class="page">

<div class="card">

<h2>Track Complaint</h2>

<p style="margin-bottom:15px;color:#6b7280;">
Enter your complaint ID to view the current status.
</p>

<div style="display:flex;gap:10px;">

<input
type="text"
id="trackId"
placeholder="Example: CMP-00001"
style="flex:1;">

<button class="primary-btn" onclick="trackComplaint()">
Track
</button>

</div>

<div id="trackResult" class="track-result"></div>

</div>

</section>


<!-- ADMIN LOGIN -->

<section id="adminLogin" class="page">

<div class="login-box">

<div class="card">

<h2>Administrator Login</h2>

<p style="color:#6b7280;margin-bottom:20px;">
Demo credentials are provided for the mini-project.
</p>

<div id="loginError" class="error"></div>

<div class="form-group">

<label>Username</label>

<input
type="text"
id="adminUsername"
placeholder="admin">

</div>

<br>

<div class="form-group">

<label>Password</label>

<input
type="password"
id="adminPassword"
placeholder="admin123">

</div>

<br>

<button class="primary-btn" onclick="adminLogin()">
Login
</button>

</div>

</div>

</section>


<!-- ADMIN DASHBOARD -->

<section id="adminDashboard" class="page">

<div class="hero">

<h1>Administrator Dashboard</h1>

<p>
Monitor, filter and manage campus complaints.
</p>

</div>


<div class="stats">

<div class="stat">
<div class="stat-title">Total</div>
<div class="stat-value" id="adminTotal">0</div>
</div>

<div class="stat">
<div class="stat-title">Submitted</div>
<div class="stat-value" id="adminSubmitted">0</div>
</div>

<div class="stat">
<div class="stat-title">In Progress</div>
<div class="stat-value" id="adminProgress">0</div>
</div>

<div class="stat">
<div class="stat-title">Resolved</div>
<div class="stat-value" id="adminResolved">0</div>
</div>

</div>


<div class="card">

<h2>Complaint Statistics</h2>

<div class="chart-container">

<canvas id="complaintChart"></canvas>

</div>

</div>


<div class="card">

<h2>Manage Complaints</h2>

<div class="filters">

<input
type="text"
id="searchBox"
placeholder="Search complaint, student or category"
onkeyup="loadComplaints()">

<select id="statusFilter" onchange="loadComplaints()">

<option value="">All Status</option>
<option>Submitted</option>
<option>In Progress</option>
<option>Resolved</option>
<option>Rejected</option>

</select>

<select id="priorityFilter" onchange="loadComplaints()">

<option value="">All Priority</option>
<option>Low</option>
<option>Medium</option>
<option>High</option>
<option>Critical</option>

</select>

</div>


<div class="table-container">

<table>

<thead>

<tr>

<th>ID</th>
<th>Student</th>
<th>Department</th>
<th>Category</th>
<th>Priority</th>
<th>Location</th>
<th>Status</th>
<th>Action</th>

</tr>

</thead>

<tbody id="complaintsTable">

</tbody>

</table>

</div>

</div>

</section>

</div>


<footer>

Cloud-Based Campus Complaint Management System
<br>
Mini Project | Python Flask | SQLite | Google Colab

</footer>


<script>

let chart = null;


/* PAGE NAVIGATION */

function showPage(pageId, button) {

document.querySelectorAll(".page").forEach(
p => p.classList.remove("active")
);

document.getElementById(pageId).classList.add("active");

document.querySelectorAll(".nav-btn").forEach(
b => b.classList.remove("active")
);

if(button) {
button.classList.add("active");
}

if(pageId === "home") {
loadHomeStats();
}

}


/* HOME STATISTICS */

async function loadHomeStats() {

try {

const response = await fetch("/api/stats");

const data = await response.json();

document.getElementById("homeTotal").textContent = data.total;

document.getElementById("homeSubmitted").textContent =
data.submitted;

document.getElementById("homeProgress").textContent =
data.in_progress;

document.getElementById("homeResolved").textContent =
data.resolved;

}
catch(error) {

console.log(error);

}

}


/* SUBMIT COMPLAINT */

document.getElementById("complaintForm")
.addEventListener("submit", async function(event) {

event.preventDefault();

const successBox =
document.getElementById("submitSuccess");

const errorBox =
document.getElementById("submitError");

successBox.style.display = "none";
errorBox.style.display = "none";


const data = {

student_name:
document.getElementById("student_name").value,

student_email:
document.getElementById("student_email").value,

department:
document.getElementById("department").value,

year:
document.getElementById("year").value,

category:
document.getElementById("category").value,

priority:
document.getElementById("priority").value,

location:
document.getElementById("location").value,

description:
document.getElementById("description").value

};


try {

const response = await fetch(
"/api/complaints",
{
method: "POST",
headers: {
"Content-Type": "application/json"
},
body: JSON.stringify(data)
}
);

const result = await response.json();

if(result.success) {

successBox.innerHTML =
"Complaint submitted successfully.<br><br>" +
"<strong>Your Complaint ID: " +
result.complaint_id +
"</strong><br><br>" +
"Please save this ID to track your complaint.";

successBox.style.display = "block";

document.getElementById("complaintForm").reset();

window.scrollTo({
top: 0,
behavior: "smooth"
});

}
else {

errorBox.textContent = result.message;

errorBox.style.display = "block";

}

}
catch(error) {

errorBox.textContent =
"Unable to submit complaint.";

errorBox.style.display = "block";

}

});


/* TRACK COMPLAINT */

async function trackComplaint() {

const id =
document.getElementById("trackId").value.trim();

const resultBox =
document.getElementById("trackResult");

if(!id) {

resultBox.innerHTML =
"<div class='error' style='display:block'>Please enter a complaint ID.</div>";

return;

}


try {

const response =
await fetch(
"/api/complaints/" +
encodeURIComponent(id)
);

const data = await response.json();


if(!response.ok) {

resultBox.innerHTML =
"<div class='error' style='display:block'>" +
data.message +
"</div>";

return;

}


let statusClass = "status-submitted";

if(data.status === "In Progress")
statusClass = "status-progress";

if(data.status === "Resolved")
statusClass = "status-resolved";

if(data.status === "Rejected")
statusClass = "status-rejected";


resultBox.innerHTML = `

<div class="card">

<h3>${data.complaint_id}</h3>

<br>

<p><strong>Student:</strong> ${data.student_name}</p>

<p><strong>Category:</strong> ${data.category}</p>

<p><strong>Priority:</strong> ${data.priority}</p>

<p><strong>Location:</strong> ${data.location}</p>

<p>
<strong>Status:</strong>
<span class="status ${statusClass}">
${data.status}
</span>
</p>

<p><strong>Description:</strong> ${data.description}</p>

<p><strong>Admin Remark:</strong>
${data.admin_remark || "No remark yet"}
</p>

<br>

<div class="timeline">

<div class="timeline-item">

<h4>Complaint Submitted</h4>

<p>${data.created_at}</p>

</div>

<div class="timeline-item">

<h4>Current Status</h4>

<p>${data.status}</p>

</div>

</div>

</div>

`;

}
catch(error) {

resultBox.innerHTML =
"<div class='error' style='display:block'>Unable to track complaint.</div>";

}

}


/* ADMIN LOGIN */

async function adminLogin() {

const username =
document.getElementById("adminUsername").value;

const password =
document.getElementById("adminPassword").value;

const errorBox =
document.getElementById("loginError");


if(username === "admin" && password === "admin123") {

showPage("adminDashboard");

loadAdminDashboard();

}
else {

errorBox.textContent =
"Invalid username or password.";

errorBox.style.display = "block";

}

}


/* ADMIN DASHBOARD */

async function loadAdminDashboard() {

await loadAdminStats();

await loadComplaints();

}


/* ADMIN STATS */

async function loadAdminStats() {

const response =
await fetch("/api/stats");

const data =
await response.json();


document.getElementById("adminTotal").textContent =
data.total;

document.getElementById("adminSubmitted").textContent =
data.submitted;

document.getElementById("adminProgress").textContent =
data.in_progress;

document.getElementById("adminResolved").textContent =
data.resolved;


/* CHART */

const ctx =
document.getElementById("complaintChart")
.getContext("2d");


if(chart) {
chart.destroy();
}


chart = new Chart(ctx, {

type: "doughnut",

data: {

labels: [
"Submitted",
"In Progress",
"Resolved",
"Rejected"
],

datasets: [{

data: [
data.submitted,
data.in_progress,
data.resolved,
data.rejected
]

}]

},

options: {

responsive: true,

plugins: {

legend: {
position: "bottom"
}

}

}

});

}


/* LOAD COMPLAINTS */

async function loadComplaints() {

const search =
document.getElementById("searchBox").value;

const status =
document.getElementById("statusFilter").value;

const priority =
document.getElementById("priorityFilter").value;


const params =
new URLSearchParams({

search: search,
status: status,
priority: priority

});


const response =
await fetch("/api/complaints?" + params.toString());

const complaints =
await response.json();


const table =
document.getElementById("complaintsTable");


if(complaints.length === 0) {

table.innerHTML = `

<tr>

<td colspan="8"
style="text-align:center;padding:30px;">

No complaints found.

</td>

</tr>

`;

return;

}


table.innerHTML = complaints.map(c => {

let statusClass =
"status-submitted";

if(c.status === "In Progress")
statusClass = "status-progress";

if(c.status === "Resolved")
statusClass = "status-resolved";

if(c.status === "Rejected")
statusClass = "status-rejected";


return `

<tr>

<td>${c.complaint_id}</td>

<td>
${c.student_name}
<br>
<small>${c.student_email}</small>
</td>

<td>${c.department}</td>

<td>${c.category}</td>

<td>${c.priority}</td>

<td>${c.location}</td>

<td>

<span class="status ${statusClass}">
${c.status}
</span>

</td>

<td>

<button
class="primary-btn"
style="padding:7px 10px;font-size:11px;"
onclick="updateComplaint('${c.complaint_id}')">

Update

</button>

</td>

</tr>

`;

}).join("");

}


/* UPDATE COMPLAINT */

async function updateComplaint(id) {

const status =
prompt(
"Enter new status:\n\nSubmitted\nIn Progress\nResolved\nRejected"
);

if(!status) return;


const validStatuses = [
"Submitted",
"In Progress",
"Resolved",
"Rejected"
];


if(!validStatuses.includes(status)) {

alert("Invalid status.");

return;

}


const remark =
prompt("Enter admin remark:");


const response =
await fetch(
"/api/complaints/" +
encodeURIComponent(id),
{

method: "PUT",

headers: {
"Content-Type": "application/json"
},

body: JSON.stringify({

status: status,
admin_remark: remark || ""

})

});


const result =
await response.json();


if(result.success) {

alert("Complaint updated successfully.");

loadAdminDashboard();

}
else {

alert(result.message);

}

}


/* INITIAL LOAD */

loadHomeStats();

</script>

</body>

</html>
"""



@app.route("/")
def home():
    return render_template_string(HTML)



@app.route("/api/complaints", methods=["POST"])
def create_complaint():

    data = request.get_json()

    required_fields = [
        "student_name",
        "student_email",
        "department",
        "year",
        "category",
        "priority",
        "location",
        "description"
    ]

    for field in required_fields:

        if not data.get(field):

            return jsonify({
                "success": False,
                "message": f"{field.replace('_', ' ').title()} is required."
            }), 400


    conn = get_db()
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO complaints
        (
            complaint_id,
            student_name,
            student_email,
            department,
            year,
            category,
            priority,
            location,
            description,
            status
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        "TEMP",
        data["student_name"],
        data["student_email"],
        data["department"],
        data["year"],
        data["category"],
        data["priority"],
        data["location"],
        data["description"],
        "Submitted"
    ))

    complaint_number = cursor.lastrowid

    complaint_id = f"CMP-{complaint_number:05d}"

    cursor.execute("""
        UPDATE complaints
        SET complaint_id = ?
        WHERE id = ?
    """, (
        complaint_id,
        complaint_number
    ))

    conn.commit()
    conn.close()


    return jsonify({
        "success": True,
        "complaint_id": complaint_id
    })



@app.route("/api/complaints", methods=["GET"])
def get_complaints():

    search = request.args.get("search", "")
    status = request.args.get("status", "")
    priority = request.args.get("priority", "")

    conn = get_db()

    query = """
        SELECT *
        FROM complaints
        WHERE 1=1
    """

    params = []


    if search:

        query += """
            AND (
                complaint_id LIKE ?
                OR student_name LIKE ?
                OR category LIKE ?
                OR department LIKE ?
                OR location LIKE ?
            )
        """

        search_value = f"%{search}%"

        params.extend([
            search_value,
            search_value,
            search_value,
            search_value,
            search_value
        ])


    if status:

        query += " AND status = ?"

        params.append(status)


    if priority:

        query += " AND priority = ?"

        params.append(priority)


    query += " ORDER BY id DESC"


    rows = conn.execute(
        query,
        params
    ).fetchall()

    conn.close()


    return jsonify([
        dict(row)
        for row in rows
    ])



@app.route("/api/complaints/<complaint_id>", methods=["GET"])
def get_single_complaint(complaint_id):

    conn = get_db()

    row = conn.execute("""
        SELECT *
        FROM complaints
        WHERE complaint_id = ?
    """, (complaint_id.upper(),)).fetchone()

    conn.close()


    if not row:

        return jsonify({
            "message": "Complaint not found."
        }), 404


    return jsonify(dict(row))



@app.route("/api/complaints/<complaint_id>", methods=["PUT"])
def update_complaint(complaint_id):

    data = request.get_json()

    status = data.get("status")
    remark = data.get("admin_remark", "")


    valid_statuses = [
        "Submitted",
        "In Progress",
        "Resolved",
        "Rejected"
    ]


    if status not in valid_statuses:

        return jsonify({
            "success": False,
            "message": "Invalid status."
        }), 400


    conn = get_db()

    cursor = conn.cursor()

    cursor.execute("""
        UPDATE complaints
        SET
            status = ?,
            admin_remark = ?,
            updated_at = CURRENT_TIMESTAMP
        WHERE complaint_id = ?
    """, (
        status,
        remark,
        complaint_id.upper()
    ))


    if cursor.rowcount == 0:

        conn.close()

        return jsonify({
            "success": False,
            "message": "Complaint not found."
        }), 404


    conn.commit()
    conn.close()


    return jsonify({
        "success": True
    })



@app.route("/api/stats")
def statistics():

    conn = get_db()

    total = conn.execute("""
        SELECT COUNT(*) FROM complaints
    """).fetchone()[0]


    submitted = conn.execute("""
        SELECT COUNT(*)
        FROM complaints
        WHERE status = 'Submitted'
    """).fetchone()[0]


    in_progress = conn.execute("""
        SELECT COUNT(*)
        FROM complaints
        WHERE status = 'In Progress'
    """).fetchone()[0]


    resolved = conn.execute("""
        SELECT COUNT(*)
        FROM complaints
        WHERE status = 'Resolved'
    """).fetchone()[0]


    rejected = conn.execute("""
        SELECT COUNT(*)
        FROM complaints
        WHERE status = 'Rejected'
    """).fetchone()[0]


    conn.close()


    return jsonify({

        "total": total,

        "submitted": submitted,

        "in_progress": in_progress,

        "resolved": resolved,

        "rejected": rejected

    })


def run_app():

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


print("Database initialized successfully.")
print("Project created at:", PROJECT_DIR)

thread = threading.Thread(
    target=run_app,
    daemon=True
)

thread.start()

time.sleep(3)

print("Flask server started successfully on port 5000.")

Database initialized successfully.
Project created at: /content/campus_complaint_system
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Flask server started successfully on port 5000.


In [7]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "TOKEN"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(5000)

print("Your Campus Complaint Management System is live!")
print()
print("Open this URL:")
print(public_url)

Your Campus Complaint Management System is live!

Open this URL:
NgrokTunnel: "https://cactus-dawn-reclaim.ngrok-free.dev" -> "http://localhost:5000"
